# Outfitly - Two-Tower Recommendation Model

This notebook implements a Pointwise Two-Tower Neural Network using PyTorch.

1. **User Tower**: Uses `UserIndex` embedding + normalized average purchase price.
2. **Item Tower**: Uses `ItemIndex` embedding + Category, Color, and Graphic embeddings + normalized price.
3. **Loss**: Binary Cross-Entropy (BCE) with negative sampling.

In [ ]:
import os
import random
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Ensure output directory exists
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Load Processed Features

We load the processed CSVs generated during Step 2 (Feature Engineering).

In [ ]:
print("Loading feature datasets...")
user_features = pd.read_csv(os.path.join(DATA_DIR, "user_features.csv"))
item_features = pd.read_csv(os.path.join(DATA_DIR, "item_features.csv"))
train_interactions = pd.read_csv(os.path.join(DATA_DIR, "train_interactions.csv"))

print(f"Users: {len(user_features)}, Items: {len(item_features)}, Interactions: {len(train_interactions)}")

# Set up metadata for embeddings
num_users = user_features["UserIndex"].max() + 1
num_items = item_features["ItemIndex"].max() + 1
num_categories = item_features["CategoryIdx"].max() + 1
num_colors = item_features["ColorIdx"].max() + 1
num_graphics = item_features["GraphicIdx"].max() + 1

print(f"Vocab sizes - Users: {num_users}, Items: {num_items}, Categories: {num_categories}")

## 2. PyTorch Dataset with Negative Sampling

For each positive interaction (purchase), we randomly sample negative items.

In [ ]:
class OutfitlyDataset(Dataset):
    def __init__(self, interactions_df, user_features_df, item_features_df, num_items, num_negatives=4):
        self.interactions = interactions_df.values
        self.num_items = num_items
        self.num_negatives = num_negatives
        
        # Precompute feature lookups for O(1) access during training
        self.user_feats = {
            row[0]: [row[1]]  # UserIndex -> [NormAvgPrice]
            for row in user_features_df[["UserIndex", "NormAvgPrice"]].values
        }
        
        self.item_feats = {
            row[0]: [row[1], row[2], row[3], row[4]] # ItemIndex -> [CategoryIdx, ColorIdx, GraphicIdx, NormPrice]
            for row in item_features_df[["ItemIndex", "CategoryIdx", "ColorIdx", "GraphicIdx", "NormPrice"]].values
        }
        
    def __len__(self):
        # Total size = Positives * (1 + num_negatives)
        return len(self.interactions) * (1 + self.num_negatives)

    def __getitem__(self, idx):
        # Determine if this sample is positive or negative
        pos_idx = idx // (1 + self.num_negatives)
        is_positive = (idx % (1 + self.num_negatives) == 0)
        
        user_idx, item_idx_pos, label_pos = self.interactions[pos_idx]
        user_idx = int(user_idx)
        
        if is_positive:
            item_idx = int(item_idx_pos)
            label = 1.0
        else:
            # Random negative sample
            item_idx = random.randint(0, self.num_items - 1)
            # Rough check to prevent accidental positive collision (ideal: check in a set)
            while item_idx == item_idx_pos:
                item_idx = random.randint(0, self.num_items - 1)
            label = 0.0
            
        # Get features
        u_feat = self.user_feats.get(user_idx, [0.0])
        i_feat = self.item_feats.get(item_idx, [0, 0, 0, 0.0])
        
        return {
            "user_idx": torch.tensor(user_idx, dtype=torch.long),
            "user_norm_price": torch.tensor(u_feat[0], dtype=torch.float),
            "item_idx": torch.tensor(item_idx, dtype=torch.long),
            "item_cat": torch.tensor(i_feat[0], dtype=torch.long),
            "item_color": torch.tensor(i_feat[1], dtype=torch.long),
            "item_graphic": torch.tensor(i_feat[2], dtype=torch.long),
            "item_norm_price": torch.tensor(i_feat[3], dtype=torch.float),
            "label": torch.tensor(label, dtype=torch.float)
        }

# For fast testing, you can sample data
SAMPLE_SIZE = min(100_000, len(train_interactions))
train_sample = train_interactions.sample(SAMPLE_SIZE, random_state=42)

dataset = OutfitlyDataset(train_sample, user_features, item_features, num_items, num_negatives=2)
dataloader = DataLoader(dataset, batch_size=512, shuffle=True, drop_last=True)
print(f"Created DataLoader with {len(dataloader)} batches.")

## 3. Define the Two-Tower Architecture

Using fully connected layers to compress dense/sparse features into 64-D vectors.

In [ ]:
class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items, num_categories, num_colors, num_graphics, emb_dim=32, out_dim=64):
        super(TwoTowerModel, self).__init__()
        
        # --- User Tower ---
        self.user_emb = nn.Embedding(num_users, emb_dim)
        # Input to dense: user_emb (32) + NormAvgPrice (1) = 33
        self.user_dense = nn.Sequential(
            nn.Linear(emb_dim + 1, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, out_dim)
        )
        
        # --- Item Tower ---
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.cat_emb = nn.Embedding(num_categories, 16)
        self.color_emb = nn.Embedding(num_colors, 16)
        self.graphic_emb = nn.Embedding(num_graphics, 16)
        # Input to dense: Item (32) + Cat (16) + Color (16) + Graphic (16) + NormPrice (1) = 81
        self.item_dense = nn.Sequential(
            nn.Linear(emb_dim + 16 * 3 + 1, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, out_dim)
        )
        
    def forward(self, batch):
        # 1. User Representaton
        u_e = self.user_emb(batch["user_idx"])
        u_in = torch.cat([u_e, batch["user_norm_price"].unsqueeze(1)], dim=1)
        user_vector = self.user_dense(u_in)  # (Batch, 64)
        
        # 2. Item Representation
        i_e = self.item_emb(batch["item_idx"])
        c_e = self.cat_emb(batch["item_cat"])
        co_e = self.color_emb(batch["item_color"])
        g_e = self.graphic_emb(batch["item_graphic"])
        i_in = torch.cat([i_e, c_e, co_e, g_e, batch["item_norm_price"].unsqueeze(1)], dim=1)
        item_vector = self.item_dense(i_in)  # (Batch, 64)
        
        # Dot product for similarity
        scores = torch.sum(user_vector * item_vector, dim=1)
        return scores

model = TwoTowerModel(num_users, num_items, num_categories, num_colors, num_graphics)
print("Model created.")

## 4. Training Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 3

print(f"Starting training on {device}...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    
    for i, batch in enumerate(dataloader):
        # Move inputs to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        
        # Forward pass computes dot product scores
        scores = model(batch)
        
        loss = criterion(scores, batch["label"])
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if i % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {i}/{len(dataloader)} | Loss: {loss.item():.4f}")
            
    print(f"Epoch {epoch+1} completed. Average Loss: {total_loss / len(dataloader):.4f}")

## 5. Save the Model

In [ ]:
model_path = os.path.join(DATA_DIR, "two_tower_model.pth")
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}.")
print("Training complete! The model is now ready for the Retrieval API.")